# Value Score Engineering

This notebook creates the core offensive player value metric for QBs, RBs, WRs, and TEs. The primary raw metric is `value_epa_total`: QB passing plus rushing EPA for quarterbacks, and rushing plus receiving EPA for running backs, wide receivers, and tight ends.

The primary comparison metric is `value_score`, which standardizes `value_epa_total` within each season-position group. That means a 2024 WR is compared only with other 2024 WRs. The notebook also keeps `value_epa_per_game` and `value_score_per_game` as supporting context, but they are not the main ranking metric because per-game scoring can overstate players with smaller availability samples.


## Load Cleaned Player-Season-Team Data

The cleaned data can contain multiple rows for a player in the same season if he changed teams. Before scoring, those rows are collapsed to one player-season so traded players are not penalized by a team-stint filter.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(expected_file="data/processed/skill_player_seasons_2016_2025.csv"):
    """Find the repo root from common VS Code/Jupyter working directories."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / expected_file).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find " + expected_file + " from working directory " + str(Path.cwd())
    )


project_root = find_project_root()
processed_dir = project_root / "data" / "processed"

sys.path.insert(0, str(project_root / "src"))
from prediction_report import MIN_VALUE_GAMES, create_player_season_value_scores

skill_season_path = processed_dir / "skill_player_seasons_2016_2025.csv"
skill_player_team = pd.read_csv(skill_season_path)

print("Raw cleaned rows:", skill_player_team.shape)
print("Unique player-seasons before collapse:", skill_player_team[["season", "player_id"]].drop_duplicates().shape[0])
skill_player_team.head()


Raw cleaned rows: (6082, 69)
Unique player-seasons before collapse: 5873


,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,birth_date,height,weight,years_exp,entry_year,rookie_year,draft_club,draft_number,college,age
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,1977-08-03,76.0,225.0,16.0,2000.0,2000.0,NE,NaN,Michigan,39
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,1979-05-12,69.0,195.0,NaN,NaN,NaN,NaN,NaN,Utah,37
2,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,1979-01-15,72.0,209.0,15.0,2001.0,2001.0,SD,NaN,Purdue,37
3,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,1980-01-09,75.0,230.0,14.0,2002.0,2002.0,NaN,NaN,Maryland,36
4,2016,00-0021206,J.McCown,Josh McCown,QB,CLE,90,165,1100,6,...,1979-07-04,76.0,218.0,14.0,2002.0,2002.0,ARI,NaN,Sam Houston State,37


## Build Value Scores

This shared helper performs the key methodological steps:

1. Collapse player-team stints into one player-season row.
2. Calculate position-specific EPA definitions.
3. Keep players with at least four games played after the collapse.
4. Standardize `value_epa_total` within season-position groups.
5. Add ranks, percentiles, and per-game supporting metrics.


In [2]:
value_scored = create_player_season_value_scores(
    skill_player_team,
    min_games=MIN_VALUE_GAMES,
)

print("Value-scored player-season rows:", value_scored.shape)
print("Duplicate player-season rows:", value_scored.duplicated(["season", "player_id"]).sum())

display(
    value_scored[[
        "season", "player_display_name", "position", "team", "teams",
        "games_played", "value_epa_total", "value_epa_per_game",
        "value_score", "value_score_per_game", "position_season_rank",
        "position_season_percentile"
    ]].head(10)
)


Value-scored player-season rows: (4753, 81)
Duplicate player-season rows: 0


,season,player_display_name,position,team,teams,games_played,value_epa_total,value_epa_per_game,value_score,value_score_per_game,position_season_rank,position_season_percentile
0,2016,Tom Brady,QB,NE,NE,12,140.632333,11.719361,2.127596,2.264787,3.0,0.960000
1,2016,Steve Smith,WR,BAL,BAL,14,37.947165,2.710512,1.390061,1.430829,18.0,0.903409
2,2016,Drew Brees,QB,NO,NO,16,105.672159,6.604510,1.491620,1.172466,6.0,0.900000
4,2016,Josh McCown,QB,CLE,CLE,5,-29.051473,-5.810295,-0.959196,-1.478825,45.0,0.120000
5,2016,Carson Palmer,QB,ARI,ARI,15,26.261320,1.750755,0.047023,0.135903,20.0,0.620000
6,2016,Antonio Gates,TE,LAC,LAC,13,11.874654,0.913435,0.282701,0.246653,28.0,0.737864
8,2016,Andre Johnson,WR,TEN,TEN,6,-2.164626,-0.360771,-0.755386,-0.861975,147.0,0.170455
9,2016,Anquan Boldin,WR,DET,DET,16,28.075429,1.754714,0.862055,0.717298,35.0,0.806818
10,2016,Jason Witten,TE,DAL,DAL,15,22.518397,1.501226,1.060906,0.788661,17.0,0.844660
12,2016,Eli Manning,QB,NYG,NYG,16,-20.212029,-1.263252,-0.798394,-0.507764,41.0,0.200000


## Method Checks

The standardized score should be centered around zero within each season-position group. The exact mean may differ by tiny floating-point amounts, but it should be very close to zero.


In [3]:
group_check = (
    value_scored
    .groupby(["season", "position"], as_index=False)
    .agg(
        players=("player_id", "count"),
        mean_value_score=("value_score", "mean"),
        std_value_score=("value_score", "std"),
        mean_total_epa=("value_epa_total", "mean"),
        mean_epa_per_game=("value_epa_per_game", "mean"),
    )
)

display(group_check.tail(16))

assert value_scored.duplicated(["season", "player_id"]).sum() == 0
assert value_scored["games_played"].ge(MIN_VALUE_GAMES).all()


,season,position,players,mean_value_score,std_value_score,mean_total_epa,mean_epa_per_game
24,2022,QB,57,-3.895519e-18,1.0,10.765987,-0.119037
25,2022,RB,133,0.000000e+00,1.0,-5.199338,-0.477183
26,2022,TE,105,2.643388e-19,1.0,6.340741,0.471773
27,2022,WR,192,4.394633e-17,1.0,11.170005,0.732306
28,2023,QB,60,-6.013708e-18,1.0,0.929047,-0.767890
29,2023,RB,118,-4.704335e-17,1.0,-9.429758,-0.723107
30,2023,TE,100,-3.774758e-17,1.0,7.211444,0.520532
31,2023,WR,188,-4.370027e-17,1.0,11.780082,0.754993
32,2024,QB,58,-1.339924e-17,1.0,18.705612,0.527522
33,2024,RB,115,8.447349e-18,1.0,-4.654944,-0.460458


## Sanity Check 2024 Rankings

These tables show the highest value scores by position for one completed season. The ranking uses `value_score`, while `value_epa_total` keeps the result interpretable in EPA units.


In [4]:
season_to_check = 2024

for pos in ["QB", "RB", "WR", "TE"]:
    print(f"Top {pos}s in {season_to_check}")
    display(
        value_scored[
            (value_scored["season"] == season_to_check) &
            (value_scored["position"] == pos)
        ][[
            "player_display_name", "team", "games_played",
            "value_epa_total", "value_epa_per_game", "value_score",
            "value_score_per_game", "position_season_rank"
        ]]
        .sort_values("value_score", ascending=False)
        .head(10)
    )


Top QBs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score,value_score_per_game,position_season_rank
4820,Josh Allen,BUF,16,187.113574,11.694598,2.753053,2.341023,1.0
4810,Lamar Jackson,BAL,17,183.931545,10.819503,2.701035,2.157571,2.0
4732,Jared Goff,DET,17,165.429827,9.731166,2.398578,1.929417,3.0
4924,Joe Burrow,CIN,17,131.179774,7.716457,1.838674,1.507061,4.0
4819,Baker Mayfield,TB,17,122.182661,7.187215,1.691593,1.396113,5.0
5254,Jayden Daniels,WAS,17,119.373123,7.021948,1.645664,1.361467,6.0
5056,Brock Purdy,SF,15,97.007326,6.467155,1.280039,1.245162,7.0
4764,Patrick Mahomes,KC,16,95.446626,5.965414,1.254526,1.139979,8.0
4889,Tua Tagovailoa,MIA,11,86.305197,7.845927,1.105086,1.534202,9.0
4836,Kyler Murray,ARI,17,83.587614,4.916918,1.060660,0.920176,10.0


Top RBs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score,value_score_per_game,position_season_rank
5163,Jahmyr Gibbs,DET,17,50.032154,2.943068,3.281628,2.734774,1.0
4724,Derrick Henry,BAL,17,49.606434,2.918026,3.256081,2.714652,2.0
4817,Saquon Barkley,PHI,16,49.461172,3.091323,3.247364,2.853899,3.0
5184,Bucky Irving,TB,17,35.450351,2.085315,2.406612,2.045559,4.0
5092,Bijan Robinson,ATL,17,25.584958,1.504998,1.814616,1.579267,5.0
4851,Ty Johnson,BUF,17,24.069822,1.415872,1.723697,1.507654,6.0
5000,James Cook,BUF,16,23.395136,1.462196,1.683211,1.544876,7.0
4759,Austin Ekeler,WAS,12,18.801500,1.566792,1.407559,1.628920,8.0
5142,Sean Tucker,TB,17,17.547465,1.032204,1.332308,1.199372,9.0
4827,Justice Hill,BAL,15,14.844525,0.989635,1.170111,1.165167,10.0


Top WRs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score,value_score_per_game,position_season_rank
4972,Amon-Ra St. Brown,DET,17,95.743892,5.631994,4.169688,3.245646,1.0
4962,Ja'Marr Chase,CIN,17,79.032173,4.648951,3.336398,2.580230,2.0
5256,Ladd McConkey,LAC,16,65.707443,4.106715,2.671992,2.213194,3.0
4862,Terry McLaurin,WAS,17,64.637547,3.802209,2.618644,2.007075,4.0
4904,Justin Jefferson,MIN,17,63.974392,3.763200,2.585577,1.980670,5.0
4864,A.J. Brown,PHI,13,61.811166,4.754705,2.477713,2.651815,6.0
5250,Brian Thomas Jr.,JAX,17,56.810445,3.341791,2.228364,1.695421,7.0
4916,Tee Higgins,CIN,12,56.401296,4.700108,2.207963,2.614858,8.0
4964,DeVonta Smith,PHI,13,56.324606,4.332662,2.204139,2.366136,9.0
5162,Puka Nacua,LA,11,54.491021,4.953729,2.112711,2.786533,10.0


Top TEs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score,value_score_per_game,position_season_rank
4740,George Kittle,SF,15,67.944928,4.529662,4.128084,3.951506,1.0
4762,Jonnu Smith,MIA,17,58.068716,3.415807,3.443435,2.843526,2.0
4805,Mark Andrews,BAL,17,53.545150,3.149715,3.129847,2.578837,3.0
5048,Trey McBride,ARI,16,49.672450,3.104528,2.861379,2.533889,4.0
5147,Tucker Kraft,GB,17,40.513069,2.383122,2.226422,1.816287,5.0
4960,Pat Freiermuth,PIT,17,35.901233,2.111837,1.906715,1.546433,6.0
4814,Mike Gesicki,CIN,16,33.767575,2.110473,1.758804,1.545076,7.0
5158,Sam LaPorta,DET,16,31.423357,1.963960,1.596295,1.399335,8.0
4685,Zach Ertz,WAS,17,30.982541,1.822502,1.565736,1.258624,9.0
5058,Isaiah Likely,BAL,15,27.706500,1.847100,1.338631,1.283092,10.0


## Per-Game Context

`value_epa_per_game` is retained because it answers a different question: how productive was the player when active? The main `value_score` rewards full-season contribution; `value_score_per_game` can identify high-rate players whose total season value was limited by missed games or smaller roles.


In [5]:
gap_cols = [
    "season", "player_display_name", "position", "team", "games_played",
    "value_epa_total", "value_epa_per_game", "value_score",
    "value_score_per_game", "value_score_gap"
]

value_scored[gap_cols].assign(
    absolute_gap=value_scored["value_score_gap"].abs()
).sort_values("absolute_gap", ascending=False).head(20)


,season,player_display_name,position,team,games_played,value_epa_total,value_epa_per_game,value_score,value_score_per_game,value_score_gap,absolute_gap
5303,2025,Tyreek Hill,WR,MIA,4,19.834595,4.958649,0.485252,3.123034,2.637781,2.637781
3741,2022,Royce Freeman,RB,HOU,4,-16.889749,-4.222437,-0.942095,-3.365340,-2.423245,2.423245
5160,2024,Rashee Rice,WR,KC,4,18.822884,4.705721,0.334204,2.618658,2.284454,2.284454
2957,2021,Maxx Williams,TE,ARI,5,15.071474,3.014295,0.651363,2.655027,2.003664,2.003664
4351,2023,Darrell Henderson,RB,LA,4,-12.708717,-3.177179,-0.214140,-2.083777,-1.869637,1.869637
2553,2020,Kenny Golladay,WR,DET,5,21.662457,4.332491,0.323285,2.177976,1.854691,1.854691
1831,2019,Donte Moncrief,WR,PIT,4,-15.651089,-3.912772,-1.462828,-3.287239,-1.824411,1.824411
3991,2022,Javonte Williams,RB,DEN,4,-12.721118,-3.180279,-0.606157,-2.428898,-1.822740,1.822740
4773,2024,Chris Godwin Jr.,WR,TB,7,37.047007,5.292430,1.242907,3.015797,1.772891,1.772891
2862,2021,Adrian Peterson,RB,TEN,4,-11.142271,-2.785568,-0.463820,-2.206875,-1.743055,1.743055


## Save Player Value Scores

The saved file is local and ignored by Git because processed data can be regenerated from the notebooks.


In [6]:
output_path = processed_dir / "player_value_scores_2016_2025.csv"
value_scored.to_csv(output_path, index=False)

print(f"Saved {len(value_scored):,} rows to {output_path}")


Saved 4,753 rows to /Users/kylelevesque/Desktop/nfl-player-value-analysis-1/data/processed/player_value_scores_2016_2025.csv
